# 00 — Data Quality

**Purpose:** Load raw MMEX data, run validation, inspect the quality report.
Save the validated DataFrame to `data/interim/transactions_validated.parquet`.

**Inputs:** `data/raw/*.mmb` or `data/raw/*.csv`  
**Outputs:** `data/interim/transactions_validated.parquet`

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from IPython.display import display

from src.utils.logging_config import setup_notebook_logging
from src.ingestion.mmex_sqlite_parser import parse_mmex_sqlite
from src.ingestion.mmex_csv_parser import parse_mmex_csv
from src.ingestion.validator import validate_transactions
from src.storage.local_writer import LocalWriter

logger = setup_notebook_logging()

In [2]:
# ── CONFIGURE: set the path to your MMEX file ─────────────────────────────────
MMB_PATH = f"C:/Users/mkkom/Mój dysk/PersonalFinance/MMEX/personal_finance.mmb"   # ← edit this
# If you have a CSV export instead, comment the line above and use:
# CSV_PATH = "../data/raw/mmex_export_YYYY-MM-DD.csv"
# ──────────────────────────────────────────────────────────────────────────────

df = parse_mmex_sqlite(MMB_PATH)
# df = parse_mmex_csv(CSV_PATH)  # uncomment for CSV path

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

2026-03-30 13:05:23 | src.ingestion.mmex_sqlite_parser | INFO     | Parsing MMEX SQLite: C:\Users\mkkom\Mój dysk\PersonalFinance\MMEX\personal_finance.mmb
2026-03-30 13:05:23 | src.ingestion.mmex_sqlite_parser | INFO     | Loaded 16468 transactions from SQLite (Void/Deleted excluded)
2026-03-30 13:05:23 | src.ingestion.mmex_sqlite_parser | INFO     | 16468 rows after split expansion
2026-03-30 13:05:23 | src.ingestion.mmex_sqlite_parser | INFO     | Sign-flipped 15130 outflow rows (10911 withdrawals, 4219 transfers)
2026-03-30 13:05:23 | src.ingestion.currency_converter | INFO     | Base currency: PLN
2026-03-30 13:05:23 | src.ingestion.currency_converter | WARNING  | Corrupted rate in MMEX DB discarded: GBP 2018-12-02 rate=4.835e+08 — fix this entry in MMEX or it will be used when NBP cache is disabled.
2026-03-30 13:05:23 | src.ingestion.currency_converter | INFO     | Loaded 93 historical rate records (5 currencies with history)
Fetching NBP rates: 100%|████████████| 3/3 [00:05<00:0

Loaded 16468 rows, 14 columns


In [3]:
# ── Step 2: Validate ──────────────────────────────────────────────────────────
report = validate_transactions(df)
report.display()

2026-03-30 13:05:29 | src.ingestion.validator        | INFO     | Validating 16468 rows
2026-03-30 13:05:29 | src.ingestion.validator        | INFO     | Date range: 2017-09-19 to 2026-03-31
2026-03-30 13:05:29 | src.ingestion.validator        | WARNING  | Multiple currencies detected: ['PLN', 'EUR', 'USD', 'GBP', 'GEL']. Awaiting user input.
2026-03-30 13:05:29 | src.ingestion.validator        | INFO     | Validation passed


DATA QUALITY REPORT
  Rows loaded:        16468
  Date range:         2017-09-19 -> 2026-03-31
  Currencies:         ['PLN', 'EUR', 'USD', 'GBP', 'GEL']
  Uncategorised txns: 0 (0.0%)
  Null accounts:      0

  Null rates per column:
    account                    0.00%
    amount                     0.00%
    amount_pln                 0.00%
    category                   0.00%
    currency                   0.00%
    date                       0.00%
    notes                      0.00%
    payee                     25.62%
    subcategory               31.84%
    to_account                74.38%
    to_amount                  0.00%
    transaction_id             0.00%
    transaction_number         0.00%
    type                       0.00%


In [4]:
# ── Step 3: Column summary ────────────────────────────────────────────────────
info_df = pd.DataFrame({
    "dtype":      df.dtypes,
    "non_null":   df.count(),
    "null_count": df.isna().sum(),
    "null_pct":   (df.isna().mean() * 100).round(2),
    "nunique":    df.nunique(),
})
display(info_df)

,dtype,non_null,null_count,null_pct,nunique
transaction_id,int64,16468,0,0.00,16468
date,datetime64[us],16468,0,0.00,2815
account,str,16468,0,0.00,47
to_account,str,4219,12249,74.38,44
payee,str,12249,4219,25.62,357
category,str,16468,0,0.00,31
subcategory,str,11224,5244,31.84,163
amount,float64,16468,0,0.00,6087
type,str,16468,0,0.00,3
notes,str,16468,0,0.00,6068


In [5]:
# ── Step 4: Descriptive statistics ───────────────────────────────────────────
display(df.describe().T.round(2))

for col in df.select_dtypes(include=["datetime64[ns]"]).columns:
    print(f"\n{col}: {df[col].min()} → {df[col].max()} ({df[col].nunique()} unique dates)")

for col in df.select_dtypes(include=["object"]).columns:
    print(f"\n{col} — top 10 values:")
    display(df[col].value_counts().head(10))

,count,mean,min,25%,50%,75%,max,std
transaction_id,16468.0,312261926232868.0625,1.0,4235.75,8414.5,12739.25,1774789606175595.0,671528437108059.25
date,16468,2022-07-16 23:45:08.088414,2017-09-19 00:00:00,2020-09-08 18:00:00,2022-10-12 00:00:00,2024-06-25 00:00:00,2026-03-31 00:00:00,NaN
amount,16468.0,-215.754054,-25000.0,-105.925,-27.0,-7.25,15000.0,1213.398621
to_amount,16468.0,336.49632,0.0,9.99,32.0,127.8675,31252.13,1376.263615
amount_pln,16468.0,-199.621558,-31312.589,-113.32,-29.99,-8.1275,46489.91886,2066.757137



date: 2017-09-19 00:00:00 → 2026-03-31 00:00:00 (2815 unique dates)

account — top 10 values:


C:\Users\mkkom\AppData\Local\Temp\ipykernel_28692\2035216555.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=["object"]).columns:


account
ING Direct            10177
Revolut                1473
ING Liquid              722
PayPal                  635
Revolut [EUR]           625
Portfel                 591
Frozen                  591
Mama                    296
ING Emergency Fund      240
Kasia                   217
Name: count, dtype: int64


to_account — top 10 values:


...

notes
                                                                                                       7203
Lime                                                                                                    256
Bolt                                                                                                    230
Uber                                                                                                    127
zwrot                                                                                                    93
Tier                                                                                                     77
Spotify                                                                                                  75
SONY XR65X90JAEP 65" LED 4K 120Hz Android TV Full Array HDMI 2.1 DVB-T2/HEVC/H.265                       62
ubezpieczenie za SONY XR65X90JAEP 65" LED 4K 120Hz Android TV Full Array HDMI 2.1 DVB-T2/HEVC/H.265      60
Multisport Classic    


transaction_number — top 10 values:


transaction_number
                                  16467
LEGO 76218 Sanctum Sanctorum;1        1
Name: count, dtype: int64


currency — top 10 values:


currency
PLN    15557
EUR      755
GBP      102
USD       51
GEL        3
Name: count, dtype: int64

In [6]:
# ── Step 5: Spot-check rows ───────────────────────────────────────────────────
display(df.sample(min(10, len(df)), random_state=42))

,transaction_id,date,account,to_account,payee,category,subcategory,amount,type,notes,to_amount,transaction_number,currency,amount_pln
10671,10928,2023-10-08,ING Direct,NaN,Inni,Gifts,NaN,-4.89,withdrawal,"1pc, You're Turtley Awesome, Pocket Turtle Hug...",4.89,,PLN,-4.8900
4639,4758,2021-01-28,ING Direct,NaN,Retailer,Electronics,Accessories,-99.99,withdrawal,TP-LINK Archer T4U AC1300 WiFi,99.99,,PLN,-99.9900
2189,2282,2019-06-19,ING Direct,NaN,FitFood,Food,Dining out,-4.19,withdrawal,,4.19,,PLN,-4.1900
6648,6825,2022-02-04,Revolut,NaN,Taxi,Transportation,Scooter fare,-5.80,withdrawal,Lime,5.80,,PLN,-5.8000
8511,8850,2022-11-21,ING Direct,NaN,Allegro,Homeneeds,Kitchenware,-57.90,withdrawal,PyrexPolska;2x blachy do piekarnika,57.90,,PLN,-57.9000
14022,1744224946704128,2025-04-09,ING Direct,NaN,Zabka,Food,Snacks,-33.40,withdrawal,,33.40,,PLN,-33.4000
8900,9052,2023-01-09,ING Direct,NaN,lunching.pl,Food,Takeaway,-39.90,withdrawal,lunching.pl; 2023.02.01-2023.01.08,39.90,,PLN,-39.9000
8139,8294,2022-10-01,MyBenefit,NaN,MyBenefit,Leisure,Activities,-117.36,withdrawal,Multisport Classic,117.36,,PLN,-117.3600
700,658,2018-05-11,ING Direct,NaN,Drogeria,Healthcare,Beauty,-44.98,withdrawal,OLIVIA GARDEN FINGER BRUSH LARGE SZCZOTKA WŁ D...,44.98,,PLN,-44.9800
11882,12188,2024-04-24,Revolut [GBP],NaN,Transport for London,Transportation,Train fare,-18.40,withdrawal,Blackfriars to Luton,18.40,,GBP,-92.4048


In [7]:
# ── Step 6: Known-issue checks ────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")

if "date" in df.columns:
    date_series = df["date"].sort_values()
    gaps = date_series.diff()
    big_gaps = gaps[gaps > pd.Timedelta(days=14)]
    if len(big_gaps):
        print(f"\nDate gaps > 14 days ({len(big_gaps)} gap(s)):")
        for idx in big_gaps.index:
            gap_start = date_series.iloc[list(date_series.index).index(idx) - 1]
            gap_end   = date_series.loc[idx]
            n_days    = (gap_end - gap_start).days
            print(f"  {gap_start.date()} → {gap_end.date()} ({n_days} days)")
    else:
        print("No date gaps > 14 days.")

Duplicate rows: 0
No date gaps > 14 days.


In [8]:
# ── Step 7: Save — only if validation passed ─────────────────────────────────
assert report.is_passing(), "Validation failed — fix critical issues before saving"

writer = LocalWriter(project_root="..")
writer.save_interim(df, "transactions_validated")
print("Saved: data/interim/transactions_validated.parquet")

2026-03-30 13:05:29 | src.storage.local_writer       | INFO     | Wrote 16468 rows to C:\Users\mkkom\MMEXtend\data\interim\transactions_validated.parquet


Saved: data/interim/transactions_validated.parquet
